## 3 — Gráficos: Visão Por Indicador

| Gráfico | Tipo | Descrição |
|--------|------|-----------|
| 6  | Barras agrupadas | Indicador selecionado por tipo de curso e ano |
| 7 | Barras horizontais | Ranking do indicador por curso (último ano) |
| 8  | Heatmap | Indicador por Curso × Ano |
| 9  | Heatmap | Todos os indicadores por Curso (último ano) |

### 3.1. Importações e configurações

In [45]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", None)

CORES_INDICADORES = {
    "TC":      "#2196F3",
    "TE":      "#F44336",
    "TR":      "#FF9800",
    "IEf":     "#4CAF50",
    "TPE":     "#9C27B0",
    "TEFAcad": "#00BCD4",
}

ROTULOS_INDICADORES = {
    "TC": "Taxa de Conclusão",
    "TE": "Taxa de Evasão",
    "TR": "Taxa de Retenção",
    "TPE": "Taxa de Permanência e Êxito",
    "IEf": "Índice de Eficiência",
    "TEFAcad": "Taxa de Efetividade Acadêmica",
}

# Indicador a analisar nos gráficos:
# Trocar para "TE", "TR", "IEf", "TPE" ou "TEFAcad" conforme necessário.
INDICADOR = "TC"


print("Indicador selecionado:", INDICADOR)

Indicador selecionado: TC


### 3.2. Carga dos dados e cálculo dos indicadores

In [46]:
df_dados_tratados = pd.read_parquet("../dados_tratados/restinga_dados_tratados.parquet")

print("Shape:", df_dados_tratados.shape)
df_dados_tratados.head(3)

Shape: (10445, 19)


,Ano,Categoria da Situação,Cor / Raça,Código da Matricula,Código do Ciclo Matricula,Data de Fim Previsto do Ciclo,Data de Inicio do Ciclo,Data de Ocorrencia da Matricula,Eixo Tecnológico,Faixa Etária,Mês De Ocorrência da Situação,Nome de Curso,Renda Familiar,Sexo,Situação de Matrícula,Subeixo Tecnológico,Tipo de Curso,Turno,Concluinte_Previsto
0,2017,Em curso,Não declarada,44681986,1207788,2014-12-22,2012-03-05,2012-03-01,Informação e Comunicação,35 a 39 anos,2018-03-05,Análise e Desenvolvimento de Sistemas,Não declarada,F,Retido,Informática,Superior-Tecnologia,Matutino,False
1,2017,Em curso,Branca,66262145,2007734,2019-12-20,2016-02-22,2016-02-01,Controle e Processos Industriais,15 a 19 anos,2016-02-01,Técnico em Eletrônica,"0,5<RFP<=1",F,Em fluxo,Elétrica,Técnico-Integrado,Matutino,False
2,2017,Em curso,Branca,66262155,2007734,2019-12-20,2016-02-22,2016-02-01,Controle e Processos Industriais,15 a 19 anos,2016-02-01,Técnico em Eletrônica,"0,5<RFP<=1",F,Em fluxo,Elétrica,Técnico-Integrado,Matutino,False


In [47]:
df_dados_tratados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10445 entries, 0 to 10444
Data columns (total 19 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   Ano                              10445 non-null  int64         
 1   Categoria da Situação            10445 non-null  object        
 2   Cor / Raça                       10445 non-null  object        
 3   Código da Matricula              10445 non-null  int64         
 4   Código do Ciclo Matricula        10445 non-null  int64         
 5   Data de Fim Previsto do Ciclo    10445 non-null  datetime64[ns]
 6   Data de Inicio do Ciclo          10445 non-null  datetime64[ns]
 7   Data de Ocorrencia da Matricula  10445 non-null  datetime64[ns]
 8   Eixo Tecnológico                 10445 non-null  object        
 9   Faixa Etária                     10445 non-null  object        
 10  Mês De Ocorrência da Situação    10445 non-null  datetime6

In [48]:
def calcular_indicadores(df, agrupamento):
    
    df_indicadores = (
        df.groupby(agrupamento)
        .agg(
            ingressantes    = ("Situação de Matrícula", lambda x: (x == "Ingressante").sum()),
            em_curso        = ("Categoria da Situação",  lambda x: (x == "Em curso").sum()),
            concluintes     = ("Categoria da Situação",  lambda x: (x == "Concluintes").sum()),
            evadidos        = ("Categoria da Situação",  lambda x: (x == "Evadidos").sum()),
            matr_atendidas  = ("Código da Matricula",    "count"),
            conc_no_prazo   = ("Situação de Matrícula",  lambda x: (x == "Concluída no prazo").sum()),
            conc_com_atraso = ("Situação de Matrícula",  lambda x: (x == "Concluída com atraso").sum()),
            abandono        = ("Situação de Matrícula",  lambda x: (x == "Abandono").sum()),
            desligados      = ("Situação de Matrícula",  lambda x: (x == "Desligada").sum()),
            transf_ext      = ("Situação de Matrícula",  lambda x: (x == "Transf. externa").sum()),
            transf_int      = ("Situação de Matrícula",  lambda x: (x == "Transf. interna").sum()),
            integralizadas  = ("Situação de Matrícula",  lambda x: (x == "Integralizada").sum()),
            conc_previstos  = ("Concluinte_Previsto",    "sum"),
            MREG            = ("Situação de Matrícula",
                               lambda x: ((x == "Em fluxo") | (x == "Ingressante")).sum()),
            MRET            = ("Situação de Matrícula",  lambda x: (x == "Retido").sum()),
        )
        .reset_index()
    )

    ma = df_indicadores["matr_atendidas"].replace(0, np.nan)

    df_indicadores["TC"] = (
        (df_indicadores["conc_no_prazo"] + df_indicadores["conc_com_atraso"]) / ma * 100
    )
    df_indicadores["TE"] = (
        (df_indicadores["abandono"] + df_indicadores["desligados"] + df_indicadores["transf_ext"])
        / ma * 100
    )
    df_indicadores["TR"] = df_indicadores["MRET"] / ma * 100

    matr_finalizadas = (
        df_indicadores["conc_no_prazo"] + df_indicadores["conc_com_atraso"]
        + df_indicadores["abandono"] + df_indicadores["desligados"]
        + df_indicadores["transf_int"] + df_indicadores["transf_ext"]
    ).replace(0, np.nan)
    df_indicadores["IEf"] = (
        (df_indicadores["conc_no_prazo"] + df_indicadores["conc_com_atraso"])
        / matr_finalizadas * 100
    )

    df_indicadores["t_matr_cont_reg"] = df_indicadores["MREG"] / ma * 100
    df_indicadores["TPE"]             = df_indicadores["TC"] + df_indicadores["t_matr_cont_reg"]
    df_indicadores["TEFAcad"]         = (
        df_indicadores["conc_no_prazo"]
        / df_indicadores["conc_previstos"].replace(0, np.nan) * 100
    )

    return df_indicadores.fillna(0).round(2)


# Calcula para diferentes granularidades
ind_ano_curso = calcular_indicadores(df_dados_tratados, ["Ano", "Nome de Curso"])
ind_ano_tipo  = calcular_indicadores(df_dados_tratados, ["Ano", "Tipo de Curso"])

# Identifica o último ano disponível (usado nos gráficos 7 e 9)
ultimo_ano = int(ind_ano_curso["Ano"].max())

print("ind_ano_curso:", ind_ano_curso.shape)
print("Último ano disponível:", ultimo_ano)

ind_ano_curso: (86, 24)
Último ano disponível: 2024


### 3.3. Gráficos

In [49]:
# 6: Indicador por tipo de curso e ano

fig_g6 = px.bar(
    ind_ano_tipo,
    x="Ano",
    y=INDICADOR,
    color="Tipo de Curso",
    barmode="group",   
    labels={"color": f"{ROTULOS_INDICADORES[INDICADOR]} (%)", "Tipo de Curso": "Tipo de Curso"},
    title=f"{ROTULOS_INDICADORES[INDICADOR]} por Tipo de Curso e Ano",
    text_auto=".1f",
)
fig_g6.update_xaxes(tickmode="linear", dtick=1)
fig_g6.show()

In [50]:
# 7 : Ranking do indicador por curso — APENAS O ÚLTIMO ANO

# Filtra somente os dados do último ano
ind_ultimo_ano_curso = ind_ano_curso[ind_ano_curso["Ano"] == ultimo_ano]

# Ordena do menor para o maior para que a barra maior fique no topo
ranking = ind_ultimo_ano_curso[["Nome de Curso", INDICADOR]].sort_values(
    INDICADOR, ascending=True
)

# Define a escala de cor: vermelho para indicadores negativos, azul para positivos
escala = "Reds" if INDICADOR in ["TE", "TR"] else "Blues"

fig_g7 = px.bar(
    ranking,
    x=INDICADOR,
    y="Nome de Curso",
    orientation="h",
    color=INDICADOR,
    color_continuous_scale=escala,
    text_auto=".1f",
    title=f" Ranking de {ROTULOS_INDICADORES[INDICADOR]} por Curso ({ultimo_ano})",
    labels={INDICADOR: f"{ROTULOS_INDICADORES[INDICADOR]} (%)", "Nome de Curso": ""},
)
fig_g7.update_layout(coloraxis_showscale=False)
fig_g7.show()

In [51]:
# 8: Heatmap do indicador por Curso × Ano
# Cada célula = valor do indicador para um curso em um ano.
# pivot_table() reorganiza o DataFrame: cursos nas linhas, anos nas colunas.

pivot = ind_ano_curso.pivot_table(
    index="Nome de Curso",
    columns="Ano",
    values=INDICADOR,
    fill_value=0,
)

escala_heatmap = "Reds" if INDICADOR in ["TE", "TR"] else "Blues"

fig_g8 = px.imshow(
    pivot,
    text_auto=".1f",
    color_continuous_scale=escala_heatmap,
    labels={"color": f"{ROTULOS_INDICADORES[INDICADOR]} (%)"},
    title=f"{ROTULOS_INDICADORES[INDICADOR]} por Curso e Ano",
    aspect="auto",
)
fig_g8.show()

In [53]:
# 9: Heatmap de todos os indicadores por curso (último ano)

todos_indicadores = ["TC", "TE", "TR", "IEf", "TPE", "TEFAcad"]

# Seleciona apenas o último ano e usa o curso como índice
pivot_todos = (
    ind_ano_curso[ind_ano_curso["Ano"] == ultimo_ano]
    .set_index("Nome de Curso")[todos_indicadores]
)

fig_g9 = px.imshow(
    pivot_todos,
    text_auto=".1f",
    color_continuous_scale="Greens",
    labels={"color": "(%)"},
    title=f"Todos os Indicadores por Curso ({ultimo_ano})",
    aspect="auto",
)
fig_g9.show()